In [ ]:
#BRUTE FORCE 

import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

path = r"C:\Users\joven\Documents\MIT Iqhack 26\Files\P3_tiny_ripple.qasm"
qc = QuantumCircuit.from_qasm_file(path)

# Statevector simulation (exact)
sv = Statevector.from_instruction(qc).data  # complex numpy array, length 2**20
probs = np.abs(sv)**2

# Get top-k bitstrings
k = 10
top = np.argpartition(probs, -k)[-k:]
top = top[np.argsort(probs[top])[::-1]]

n = qc.num_qubits
def idx_to_bitstring(i, n):
    # Qiskit uses little-endian indexing for the statevector basis.
    # This returns a string ordered q[n-1]...q[0] (common display).
    return format(i, f"0{n}b")

print(f"num_qubits = {n}, statevector_len = {len(sv)}")

for rank, i in enumerate(top, 1):
    print(
        f"{rank:2d}. |{idx_to_bitstring(i,n)}⟩  "
        f"p={probs[i]:.12f}  amp={sv[i]}"
    )

i_max = top[0]
print("\nDOMINANT BITSTRING:")
print(f"|{idx_to_bitstring(i_max,n)}⟩ with probability {probs[i_max]:.12f}")


In [ ]:
# MPS - not viable due to high bond dimension 

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

qc = QuantumCircuit.from_qasm_str(qasm)

qc_m = qc.copy()
qc_m.measure_all()

sim = AerSimulator(method="matrix_product_state")
sim.set_options(
    matrix_product_state_max_bond_dimension=128,
    matrix_product_state_truncation_threshold=1e-12,
)

res = sim.run(qc_m, shots=200, seed_simulator=1, seed_transpiler=1).result()
counts = res.get_counts()

dominant = max(counts, key=counts.get)
print("Dominant:", dominant, "freq:", counts[dominant]/200)


In [4]:
import time
from qiskit import QuantumCircuit

t0 = time.time()
qc = QuantumCircuit.from_qasm_str(qasm)
print("Parsed QASM in", round(time.time()-t0, 2), "s")

print("num_qubits:", qc.num_qubits)
print("depth:", qc.depth())
print("size (ops):", qc.size())
print("op counts:", qc.count_ops())

import numpy as np

cz_pairs = []
for inst, qargs, cargs in qc.data:
    if inst.name == "cz":
        a = qc.find_bit(qargs[0]).index
        b = qc.find_bit(qargs[1]).index
        cz_pairs.append((a, b))

dists = np.array([abs(a-b) for a,b in cz_pairs])
print("CZ count:", len(dists))
print("mean |a-b|:", dists.mean())
print("median |a-b|:", np.median(dists))
print("max |a-b|:", dists.max())
print("fraction with |a-b|>=10:", np.mean(dists >= 10))
print("fraction with |a-b|>=15:", np.mean(dists >= 15))



Parsed QASM in 0.02 s
num_qubits: 30
depth: 93
size (ops): 2073
op counts: OrderedDict([('u3', 1392), ('cz', 681)])
CZ count: 681
mean |a-b|: 10.81791483113069
median |a-b|: 10.0
max |a-b|: 28
fraction with |a-b|>=10: 0.5095447870778267
fraction with |a-b|>=15: 0.2907488986784141


C:\Users\joven\AppData\Local\Temp\ipykernel_21664\1769201670.py:16: DeprecationWarning: Treating CircuitInstruction as an iterable is deprecated legacy behavior since Qiskit 1.2, and will be removed in Qiskit 3.0. Instead, use the `operation`, `qubits` and `clbits` named attributes.
  for inst, qargs, cargs in qc.data:


In [6]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

# 1) Load circuit
qc = QuantumCircuit.from_qasm_str(qasm)

# 2) Use Aer statevector (no compilation tricks)
sim = AerSimulator(method="statevector")
sim.set_options(precision="single")  # BIG memory win

# 3) Minimal transpile (avoid “optimizing compilation”)
tqc = transpile(qc, sim, optimization_level=0)

# 4) Ask Aer to save the final statevector
tqc.save_statevector()

# 5) Run
res = sim.run(tqc).result()

# 6) Extract statevector and find max-probability basis state
sv = np.asarray(res.get_statevector(tqc), dtype=np.complex64)  # ensure single precision array

# argmax of |amp|^2 without allocating a full probs array
i_max = int(np.argmax((sv.real * sv.real) + (sv.imag * sv.imag)))

n = qc.num_qubits
bit = format(i_max, f"0{n}b")  # displayed as q[n-1]...q[0] in Qiskit convention
p_max = float((sv.real[i_max]**2 + sv.imag[i_max]**2))

print("Dominant bitstring:", bit)
print("Probability:", p_max)


Dominant bitstring: 001110001111101100001101010001
Probability: 0.20000313242403323
